# Loopcom Yiddish Whisper fine-tune — Kaggle free-GPU run

Pushed and started by `kaggle-run.ts kernel-push`. The attached dataset
(`kernel-metadata.json` -> `dataset_sources`) is `build-dataset.ts`'s output
folder, self-contained: `clips/`, `train.jsonl`, `eval.jsonl`,
`manifest.json`, plus copies of `train.py`, `baseline.py` and
`requirements-kaggle.txt` that `kaggle-run.ts dataset-push` staged alongside
the audio.

This notebook installs deps (torch is already present on the Kaggle GPU
image — see `requirements-kaggle.txt`'s own comment for why it is never
reinstalled here), runs `train.py` with the 16GB-card defaults (batch 4,
grad-accum 8, fp16/bf16 auto-detected, LoRA by default), and prints
`report.json` at the end. `kaggle-run.ts download` pulls `/kaggle/working/`
back afterwards.

In [ ]:
import glob, os, shutil, subprocess, sys

# Kaggle mounts every attached dataset under /kaggle/input/<slug>/.
# TWO are attached on purpose:
#   * the DATA set  — clips/ + train.jsonl + eval.jsonl + manifest.json (GBs)
#   * the CODE set  — train.py, baseline.py, requirements-kaggle.txt (KBs)
# They are split because a one-line fix to train.py used to mean re-uploading
# the whole multi-GB audio set just to change a script. Code now re-versions in
# seconds. If only one dataset is attached (older runs), fall back to it for
# both roles so this notebook still works.
dirs = [p for p in glob.glob('/kaggle/input/*') if os.path.isdir(p)]
assert dirs, 'no dataset attached under /kaggle/input - check kernel-metadata.json dataset_sources'

def has(d, name):
    return os.path.exists(os.path.join(d, name))

# The DATA set also carries a (now stale) train.py from the first upload, and
# picking that one silently ran the OLD code while the log happily said it had
# staged a script - the 2026-09-18 run lost a GPU session to exactly this.
# So the CODE set is the one with train.py and WITHOUT train.jsonl; only fall
# back to "any dir with train.py" when no such dir exists.
CODE_DIR = next((d for d in dirs if has(d, 'train.py') and not has(d, 'train.jsonl')), None)
if CODE_DIR is None:
    CODE_DIR = next((d for d in dirs if has(d, 'train.py')), None)
    print('WARNING: falling back to a dataset that also holds the data; train.py may be stale:', CODE_DIR)
DATASET_DIR = next((d for d in dirs if has(d, 'train.jsonl')), None)
assert CODE_DIR, f'no attached dataset contains train.py: {dirs}'
assert DATASET_DIR, f'no attached dataset contains train.jsonl: {dirs}'
print('code dir   :', CODE_DIR)
print('dataset dir:', DATASET_DIR)
print('dataset contents:', sorted(os.listdir(DATASET_DIR))[:20])

os.chdir('/kaggle/working')
for name in ('train.py', 'baseline.py', 'requirements-kaggle.txt'):
    src = os.path.join(CODE_DIR, name)
    assert os.path.exists(src), f'{name} missing from the code dataset ({CODE_DIR})'
    shutil.copy(src, name)
print('staged:', sorted(n for n in os.listdir('.') if n.endswith(('.py', '.txt'))))


In [ ]:
# torch/torchaudio are already on the Kaggle GPU image — requirements-kaggle.txt
# deliberately omits them (see that file's comment).
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt'], check=True)


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('bf16 supported:', torch.cuda.is_bf16_supported())


In [ ]:
# 16GB-card defaults live IN train.py (batch 4, grad-accum 8, max-steps 2000,
# eval every 250 steps, bf16-if-supported else fp16) — nothing to override
# here for a T4/P100 session. Pass --max-steps/--batch-size to shorten a run
# that would not fit in Kaggle's 12h session limit.
subprocess.run(
    [sys.executable, 'train.py', '--dataset', DATASET_DIR, '--output-dir', '/kaggle/working/out'],
    check=True,
)


In [ ]:
import json
with open('report.json') as f:
    report = json.load(f)
print(json.dumps(report, indent=2))
